# Acquiring data from APIs
We use the **World Bank Indicators API**: free, public and no key required.
Documentation: https://datahelpdesk.worldbank.org/knowledgebase/articles/889392

## JSON in Python

In [ ]:
import json

raw = '{"iso3": "KEN", "gdp": [1.2, 1.3]}'
data = json.loads(raw)
print(type(data))
print(data["iso3"], data["gdp"][0])
print(json.dumps(data, indent=2))

## Sending a request

In [ ]:
import requests

BASE = "https://api.worldbank.org/v2"
INDICATOR = "NE.EXP.GNFS.ZS"  # exports % GDP
path = f"country/KEN/indicator/{INDICATOR}"
params = {"format": "json",
          "date": "2015:2023"}
resp = requests.get(f"{BASE}/{path}",
                    params=params,
                    timeout=30)
print(resp.status_code)
print(resp.url)

## Inspecting the response

In [ ]:
payload = resp.json()
meta, records = payload
print(meta)
first = records[0]
print(first["country"]["value"],
      first["date"], first["value"])

## From JSON to a DataFrame

In [ ]:
import pandas as pd

rows = [
    {"iso3": r["countryiso3code"],
     "year": int(r["date"]),
     "value": r["value"]}
    for r in records
]
df = pd.DataFrame(rows).sort_values("year")
df.tail()

## A reusable, defensive function

In [ ]:
def fetch_indicator(countries, indicator,
                    start, end):
    path = (f"country/{';'.join(countries)}"
            f"/indicator/{indicator}")
    params = {"format": "json",
              "date": f"{start}:{end}",
              "per_page": 1000}
    resp = requests.get(f"{BASE}/{path}",
                        params=params,
                        timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    if len(payload) < 2 or not payload[1]:
        raise ValueError(f"API error: {payload}")
    return payload[1]

In [ ]:
recs = fetch_indicator(["KEN", "NGA", "ZAF"],
                       "NY.GDP.MKTP.CD",
                       2019, 2023)
print(len(recs), "records")

# 200 OK does not always mean success
bad = requests.get(f"{BASE}/country/XXX/"
                   "indicator/NY.GDP.MKTP.CD",
                   params={"format": "json"})
print(bad.status_code, bad.json())

## Falling back to a cached copy

In [ ]:
from pathlib import Path

CACHE = Path("../../data/cache")

def load_cached(name):
    path = CACHE / name
    with open(path, encoding="utf-8") as f:
        return json.load(f)[1]

In [ ]:
try:
    recs = fetch_indicator(["KEN"],
                           INDICATOR, 2015, 2023)
except requests.RequestException as e:
    print("API unavailable, using cache:", e)
    recs = load_cached(
        "wb_exports_pct_gdp.json")
print(len(recs), "records")